# Accessing Commissioning Data with ROSALIA 

## Introduction

This notebook demonstrates the use of ROSALIA to search data in MAST for commissioning data products, and then performing quick analyses.

This tutorial is based on been adapted from the Nexus tutorial [Streaming Products from S3 During Commissioning](https://github.com/spacetelescope/roman_notebooks/blob/main/notebooks/working_with_asdf/working_with_asdf.ipynb), with additional information added to support streaming from MAST search results.

## Imports

In [1]:
import numpy as np
import rosalia as rs
import matplotlib.pyplot as plt
import astropy.units as u
from astropy.time import Time
from astropy.coordinates import SkyCoord  # High-level coordinates

In [2]:
# products = rs.mast.query.roman_query(fileSetName="r01023*wfi*f062*", file_suffix="_cal")
# rs.utils.save_dict(products, "test_query.pkl")
products = rs.utils.load_dict("test_query.pkl")

In [5]:
exposure = rs.exposure(products[0])


2026-11-21 17:58:34 INFO     MAST API token accepted, welcome Alejandro Serrano Borlaff


INFO: MAST API token accepted, welcome Alejandro Serrano Borlaff [astroquery.mast.auth]


Retrieving exposure : 100%|██████████| 18/18 [05:23<00:00, 17.98s/it]

Currently estimated through pysiaf and quaternion. This is a temporary fix until MAST fixes the RA_TARG, DEC_TARG, PA keywords in the Roman/WFI fits files.


In [6]:
masks = rs.gnu.detect_sources(exposure.DATA)

  0%|          | 0/18 [00:00<?, ?it/s]

: 

In [ ]:
"""
Example of criteria for a roman_query using astroquery.mast.query_criteria
search = {'program': 1020,
         'observation': 5,
         'pass': 2, 
         #'exposure_start_time': '2026-09-15T02:01:47.4080000',
         'product_type': 'l2', 
         'detector': 'WFI06',
         # 'optical_element': 'F158',
         'exposure_type': "WFI_IMAGE"}

Here we use a region criteria, based on a set of coordinates and a search radius.
"""

coordinates=SkyCoord(229.64044*u.deg, -3.58747*u.deg, frame="icrs")
radius=10*u.arcsec

# query = roman_query_criteria(search=search, file_suffix="_cal")
results, products, exposures, downloads = rs.mast.query.roman_query(coordinates=coordinates, radius=radius, file_suffix="_cal")
data_stream = rs.mast.query.stream_roman_mast(products=products, row=0)

roman_exposure = rs.core.exposure(data_stream)
roman_exposure.save_drz(outname="dragon_drz.fits", resolution=0.11)

In [ ]:
import os
os.environ["MAST_API_TOKEN"]
token = os.getenv("MAST_API_TOKEN") 
print(token)

In [ ]:
# Let's demostrate the use of ROSALIA to estimate stray-light
# During commissioning, the astrometry might be off by a substantial angular distance. 
# Use only for testing purposes. 
roman_exposure.straylight()

In [ ]:
# This line generates a mosaic FITS file with the datasets included in your exposure object, useful to visualize the results.
roman_exposure.save_drz(outname="dragon_drz.fits", resolution=0.11)

In [ ]:
# This line generates a DS9 file with the nearest stars around the image. 
roman_exposure.make_close_stars_ds9_region(radius=5, output="nearby_stars.reg")

## About this Notebook

**Author:** Alejandro S. Borlaff - NASA Ames Research Center (Code S) - a.s.borlaff@nasa.gov 

**Updated on** September 18, 2026. 